# Hakam — training (Colab GPU)

Runs the baseline and both experiments for §7.

| Run | What it answers |
|---|---|
| `probe` | Baseline: how far do frozen Kinetics features get? |
| `finetune`, crop | **Experiment 1** — does fine-tuning beat frozen? |
| `finetune`, resize | **Experiment 2** — does keeping the full frame beat centre-cropping? |

Experiment 2 exists because the processor centre-crops 224 from a 398-wide frame,
discarding 87px each side. Measured on 57 clips, **12% of contact points fall
outside that crop entirely** — for those, the model classifies a clip that no
longer contains the foul.

**Runtime → Change runtime type → GPU** first. NDA: video stays in the ephemeral
runtime; only metrics and checkpoints go to Drive.

In [ ]:
# 1. GPU check. On CPU this notebook is hours instead of ~1 hour, so fail loudly.
import torch

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU."
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Code + dependencies.
%cd /content
!git clone https://github.com/FerasMad/hakam.git 2>/dev/null || (cd hakam && git pull -q)
%cd /content/hakam
!pip install -q transformers SoccerNet opencv-python-headless

import transformers
print("transformers", transformers.__version__)

In [ ]:
# 3. Dataset into the runtime. ~3.3 GB, about 4 minutes.
#
#    downloadDataTask takes the password as an ARGUMENT. Setting only the
#    .password attribute (what downloadGames reads) leaves the request
#    unauthenticated and fails with HTTP 401.
from pathlib import Path

from SoccerNet.Downloader import SoccerNetDownloader

try:
    from google.colab import userdata

    pw = userdata.get("SOCCERNET_PASSWORD")
except Exception:
    from getpass import getpass

    pw = getpass("SOCCERNET_PASSWORD secret not found - enter password: ")

print(f"password loaded: {len(pw)} characters")
root = Path("/content/hakam/data/mvfouls")
d = SoccerNetDownloader(LocalDirectory=str(root))
d.password = pw
d.downloadDataTask(task="mvfouls", split=["train", "valid", "test"], password=pw)

gb = sum(p.stat().st_size for p in root.rglob("*") if p.is_file()) / 1e9
print(f"\n{gb:.2f} GB downloaded")
if gb < 0.5:
    raise SystemExit("download failed - if you saw HTTP 401 the password is wrong")

In [ ]:
# 4. Layout. The downloader nests under mvfouls/mvfouls/ and the zips extract
#    FLAT, so every split writes action_0, action_1, ... into one directory and
#    silently overwrites the others. This script handles both.
!python scripts/normalise_colab_layout.py

In [ ]:
# 5. Frame cache: last clip per action, ~3.5 min for all three splits.
#
#    The last clip is a close-up 95% of the time at median 1.8x replay speed -
#    the clearest view of the contact - and using only it cuts the corpus from
#    6,621 clips to 2,916. Cached at native 224x398, BEFORE the processor, so
#    crop vs resize stays a flag rather than a second decode pass.
!python scripts/cache_frames.py --splits train valid test

In [ ]:
# 6. BASELINE - frozen backbone, logistic head. Minutes.
#    Expect ~0.55-0.58 balanced accuracy: deliberately weak, and the control
#    that makes fine-tuning's gain measurable.
!python -m src.models.train --mode probe --stage card --geometry crop \
    --batch-size 32 --num-workers 2 --name card_probe_crop

In [ ]:
# 7. EXPERIMENT 1 - fine-tuned, same centre crop. The one variable is frozen
#    versus fine-tuned. ~30-40 min.
!python -m src.models.train --mode finetune --stage card --geometry crop \
    --augment mild_aug_v1 --freeze-blocks 6 --epochs 8 \
    --batch-size 8 --num-workers 2 --save --name card_finetune_crop

In [ ]:
# 8. EXPERIMENT 2 - same fine-tuning, full frame instead of centre crop.
#    The one variable is geometry. ~30-40 min.
!python -m src.models.train --mode finetune --stage card --geometry resize \
    --augment mild_aug_v1 --freeze-blocks 6 --epochs 8 \
    --batch-size 8 --num-workers 2 --save --name card_finetune_resize

In [ ]:
# 9. Comparison table for the report.
import json
from pathlib import Path

import pandas as pd

rows = []
for f in sorted(Path("artifacts/runs").glob("*/metrics.json")):
    m = json.loads(f.read_text())
    rec = next((v for k, v in m.items() if k.startswith("at_recall")), {})
    rows.append({
        "run": m["run"],
        "mode": m["mode"],
        "geometry": m["geometry"],
        "balanced_acc": m["argmax"]["balanced_accuracy"],
        "recall@thr": rec.get("recall_card"),
        "review_load": rec.get("review_load"),
        "selective_acc": m["selective"]["selective_accuracy"],
        "minutes": m["minutes"],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print("\nAlways name the stage. Recall is never quoted without its review load.")

In [ ]:
# 10. Metrics and checkpoints to Drive. No video, ever.
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
dest = Path("/content/drive/MyDrive/hakam/runs")
dest.mkdir(parents=True, exist_ok=True)

for run in sorted(Path("artifacts/runs").glob("*")):
    if run.is_dir():
        shutil.copytree(run, dest / run.name, dirs_exist_ok=True)
        print(f"{run.name}: {[p.name for p in run.iterdir()]}")

print(f"\ncopied to {dest}")